In [ ]:
import json, copy, time
import numpy as np, pandas as pd
import scipy.stats as st

N_SIMS = 100
RATES = {'1x_65mo': 65.1552, '2x_32mo': 31.8087, '3x_21mo': 20.6848}
RES = ('antigen', 'allele', 'eplet')
ETH_LABELS = {1:'Caucasian', 2:'Afroamerican', 4:'Latin', 5:'Asian', 6:'AmInd', 7:'PacIsl'}

DEF_CELLS = list(range(0, 8)) + [10]
def load_ns(nbpath, def_cells=DEF_CELLS):
    _nb = json.load(open(nbpath)); g = {}
    for i in def_cells:
        exec(compile(''.join(_nb['cells'][i]['source']), f'<{nbpath}:{i}>', 'exec'), g)
    return g

PLAIN = load_ns('../ABO+DSA/10loci_simulation_v2.ipynb')          # plain: no HLA threshold (k_opt=0)
THR   = load_ns('../ABO+DSA+HLA/10loci_simulation_v2.ipynb')      # threshold: k_graph/k_opt diagonal
print('loaded build_results_table:', callable(PLAIN.get('build_results_table')), callable(THR.get('build_results_table')))

In [ ]:
# run all 100 sims for one (paradigm, scenario, rate), returns a list of per sim result dicts
_cache = {}
def _sim(ns, key, s):
    ck=(key,s)
    if ck not in _cache:
        sd = ns['load_sim_data'](s); ws = ns['build_weights_10loci'](sd)
        _cache[ck]=(sd,ws)
    return _cache[ck]

def run_cfg(paradigm, res, mp, nsims=N_SIMS):
    ns = PLAIN if paradigm=='plain' else THR
    p = copy.deepcopy(ns['SIM_PARAMS']); p['MEAN_PATIENCE'] = mp
    out=[]
    for s in range(nsims):
        sd, ws = _sim(ns, paradigm, s)
        if paradigm=='plain':
            out.append(ns['run_simulation'](s, res, sd, ws, p, ns['df_pat']))
        else:
            out.append(ns['run_simulation'](s, res, res, sd, ws, p, ns['df_pat']))  # diagonal
    return out

def relabel(df):
    df = df.copy()
    df['Ethnicity(s)'] = df['Ethnicity(s)'].map(lambda e: ETH_LABELS.get(e, e))
    return df

In [ ]:
# Build the full table per ethnicity for every (paradigm, scenario, rate)
t0=time.time(); TABLES={}   
for paradigm in ('plain','threshold'):
    ns = PLAIN if paradigm=='plain' else THR
    for res in RES:
        for rk, mp in RATES.items():
            results = run_cfg(paradigm, res, mp)
            TABLES[(paradigm,res,rk)] = relabel(ns['build_results_table'](results))
            print(f'  {paradigm:9s} {res:8s} {rk:8s}  ({time.time()-t0:.0f}s)')
print('done', len(TABLES), 'tables')

In [ ]:
# Preview: show the entire population F/L across rates per base case, then display a couple of full tables
summary=[]
for paradigm in ('plain','threshold'):
    for res in RES:
        row={'Paradigm':paradigm,'Scenario':res if paradigm=='plain' else f'{res}/{res}'}
        for rk in RATES:
            ep = TABLES[(paradigm,res,rk)].iloc[-1]  # Entire Population row
            row[f'F {rk}']=ep['F(s) (Matched)']; row[f'L {rk}']=ep['L(s) (Left Unmatched)']
        summary.append(row)
display(pd.DataFrame(summary))
# example full table
display(TABLES[('threshold','eplet','2x_32mo')])

In [ ]:
# SAVE: one sheet per base case
from pathlib import Path
OUT = Path('departure_sensitivity_10loci.xlsx')
def sheet_name(paradigm,res):
    return (f'plain_{res[:6]}' if paradigm=='plain' else f'thr_{res[:4]}_{res[:4]}')[:31]
with pd.ExcelWriter(OUT, engine='openpyxl') as w:
    for paradigm in ('plain','threshold'):
        for res in RES:
            sh = sheet_name(paradigm,res); startrow=0
            for rk, mp in RATES.items():
                # label row
                pd.DataFrame({f'{paradigm}  {res if paradigm=="plain" else res+"/"+res}  —  rate {rk}  (MEAN_PATIENCE={mp:.2f} mo)':[]}
                             ).to_excel(w, sheet_name=sh, startrow=startrow, index=False)
                TABLES[(paradigm,res,rk)].to_excel(w, sheet_name=sh, startrow=startrow+1, index=False)
                startrow += len(TABLES[(paradigm,res,rk)]) + 4
print('Saved ->', OUT)